# DQN Atari: Ivan's Task 3 – Training Pipeline and Hyperparameter Experiments

## Project overview

This notebook belongs to our group project: a Deep Q-Network (DQN) agent that learns to play the Atari game **Q\*bert** (`ALE/Qbert-v5`) with PyTorch and Gymnasium.

My part of the project (Task 3) is the **training pipeline**: the training loop in `train.py`, the logging, the saving of models and plots, and **ten hyperparameter experiments**.

The agent uses:

- a convolutional neural network (CNN) that looks at 4 stacked grayscale frames,
- experience replay (a memory of past moves),
- a target network (a frozen copy that gives steadier goals),
- epsilon-greedy action choice (sometimes random, sometimes the best-looking move).

## Objective

Find out how six hyperparameters change how well the agent learns: **learning rate, gamma, batch size, epsilon start, epsilon end and epsilon decay**. Each experiment changes one of them from a baseline and keeps everything else the same.

## What this notebook does

1. Checks the machine (GPU or CPU).
2. Connects Google Drive so nothing is lost if Colab disconnects.
3. Gets my branch of the project from GitHub and installs everything with **uv**.
4. Loads and checks my ten experiment settings.
5. Runs the tests and a short smoke test of the training pipeline.
6. Loads my ten finished training runs (or trains them again if you choose).
7. Tests every saved model with the same ten games, then ranks them.
8. Draws comparison plots, shows the gameplay clip, and builds a final submission folder.

**Honest note about the training:** my ten runs were trained once, on a normal computer **CPU** (33 to 68 minutes each, 100,000 steps), not on a Colab GPU. This notebook therefore loads those saved results by default. Training all ten again is optional and is switched on in section 8.

# 1. Check the machine

In [ ]:
# Check which computer we are running on (GPU is optional for this notebook)
import os, sys, json, csv, shutil, subprocess, tempfile, zipfile, time, platform
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("Running in Google Colab:", IN_COLAB)
print("Python version:", platform.python_version())

try:
    import torch
    print("PyTorch version:", torch.__version__)
    print("CUDA (GPU) available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU name:", torch.cuda.get_device_name(0))
    else:
        print("No GPU found. That is fine: loading saved results and evaluation also work on CPU.")
except ImportError:
    print("PyTorch is not installed in this Python yet. It is installed in section 4 with uv.")

# 2. Connect Google Drive

Files saved only inside a Colab session can disappear when the session ends. To keep my results, I connect Google Drive and use one permanent folder for them.

Outside Colab (for example on my laptop), a temporary folder is used instead, so this notebook can also be run locally.

In [ ]:
# Connect Google Drive (Colab) or use a local folder
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    STORAGE_ROOT = Path("/content/drive/MyDrive/dqn-experiments-ivan")
else:
    STORAGE_ROOT = Path(tempfile.gettempdir()) / "dqn-experiments-ivan"

STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
print("Permanent storage folder:", STORAGE_ROOT)

# 3. Get the Project from GitHub

The notebook downloads the group repository, **branch `Ivan`**. This branch has my training code, my ten experiment settings, my saved models, logs and plots, the evaluation script and the gameplay demo.

I use the group's tested code instead of rewriting it in Colab, so the notebook and the repository always agree.

If the notebook is opened from inside the project folder, that folder is used directly.

In [ ]:
# Clone my branch of the group repository (or reuse the current folder)
REPO_URL = "https://github.com/Hikma-codes/Formative-3---DQN-Atari-Project.git"
BRANCH = "Ivan"

if (Path.cwd() / "train.py").exists():
    REPO_DIR = Path.cwd()
    print("Using the project folder I am already in.")
else:
    REPO_DIR = Path("/content/Formative-3---DQN-Atari-Project")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    else:
        print("Repository already exists. Reusing it.")

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print("Project directory:", REPO_DIR)
print("Main files:", sorted(p.name for p in REPO_DIR.iterdir() if p.suffix in {".py", ".md", ".json", ".toml"})[:25])

# 4. Install the Dependencies with uv

The project rule is to use **uv** only (no pip). uv reads `pyproject.toml` and `uv.lock` and installs the exact package versions the group tested:

- PyTorch (neural network),
- Gymnasium and ALE-Py (the Atari game),
- NumPy, Matplotlib, OpenCV.

`--frozen` means "use the lockfile exactly as it is".

In [ ]:
# Install uv and the locked dependencies (needed in Colab; already done on my laptop)
def run(cmd, **kwargs):
    return subprocess.run(cmd, check=True, **kwargs)

if IN_COLAB:
    run("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True)
    os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
    run(["uv", "sync", "--frozen"])
else:
    print("Not in Colab: skipping the install, using the existing environment.")

run(["uv", "--version"])

# 5. Load and Check My Ten Experiment Settings

Each experiment changes **one** hyperparameter from the baseline (`TASK3_EXP01`):

| Experiments | What changes |
| --- | --- |
| EXP02, EXP03 | learning rate (smaller, bigger) |
| EXP04, EXP05 | gamma (0.95, 0.995) |
| EXP06, EXP07 | batch size (64, 16) |
| EXP08 | epsilon start (0.9) |
| EXP09 | epsilon end (0.05) |
| EXP10 | epsilon decay (50,000 steps) |

The settings are read from `ivan_experiments.json` in the repository, so the exact values used for training can be reproduced.

In [ ]:
# A small helper to show a table (nice HTML in Colab/Jupyter, plain text otherwise)
def show_table(rows, columns):
    try:
        from IPython.display import HTML, display
        head = "".join(f"<th>{c}</th>" for c in columns)
        body = "".join("<tr>" + "".join(f"<td>{row[c]}</td>" for c in columns) + "</tr>" for row in rows)
        display(HTML(f"<table><tr>{head}</tr>{body}</table>"))
    except ImportError:
        print(" | ".join(columns))
        for row in rows:
            print(" | ".join(str(row[c]) for c in columns))

In [ ]:
# Load and validate the experiment settings
EXPERIMENTS = json.loads((REPO_DIR / "ivan_experiments.json").read_text(encoding="utf-8"))
HYPERPARAMETERS = ["learning_rate", "gamma", "batch_size", "epsilon_start", "epsilon_end", "epsilon_decay"]

ids = [e["experiment_id"] for e in EXPERIMENTS]
combos = {tuple(e[h] for h in HYPERPARAMETERS) for e in EXPERIMENTS}

assert len(EXPERIMENTS) == 10, "We expected exactly 10 experiments."
assert len(set(ids)) == 10, "Duplicate experiment IDs found."
assert len(combos) == 10, "Each experiment needs a different hyperparameter combination."
assert all(e["member"] == "ivan" for e in EXPERIMENTS)

print("Experiments:", len(EXPERIMENTS), "| unique IDs:", len(set(ids)), "| unique combinations:", len(combos))
show_table(EXPERIMENTS, ["experiment_id"] + HYPERPARAMETERS)

# 6. Test the Implementation

Before trusting any result, the group's tests check that:

- the Q\*bert environment starts and gives observations of the right shape,
- the CNN gives one Q-value per action,
- the replay buffer stores and rebuilds moves correctly,
- one learning step really changes the network.

A failed test is printed, but the notebook keeps going so you can see everything.

In [ ]:
# Run the three test files with uv
RUN_TESTS = True

if RUN_TESTS:
    for test_file in ["test_environment.py", "test_dqn.py", "test_learning.py"]:
        result = subprocess.run(["uv", "run", "python", test_file], capture_output=True, text=True)
        print(f"{test_file}: {'PASSED' if result.returncode == 0 else 'FAILED'}")
        if result.returncode != 0:
            print((result.stdout + result.stderr)[-1200:])
else:
    print("Tests skipped (RUN_TESTS is False).")

# 7. Smoke Test of the Training Pipeline

A smoke test is a very short training run (100 steps) that checks the whole pipeline works: the agent plays, the network trains, a model and a plot are saved.

Its reward means nothing. It only proves the pipeline is not broken before spending hours on real training. The smoke test is saved in its own `smoke` folder, so it never mixes with the real experiments.

In [ ]:
# Short training run using train.py with a config file
os.environ["MPLBACKEND"] = "Agg"   # lets the training script save plots without a screen

smoke_config = {
    "experiment_id": "IVAN_SMOKE", "member": "smoke",
    "max_steps": 100, "learning_starts": 8, "replay_capacity": 64, "batch_size": 4,
    "train_frequency": 4, "target_update_frequency": 20, "checkpoint_frequency": 100,
    "log_frequency": 20, "device": "auto", "output_root": str(STORAGE_ROOT),
}
smoke_config_path = Path(tempfile.gettempdir()) / "ivan_smoke_config.json"
smoke_config_path.write_text(json.dumps(smoke_config, indent=2), encoding="utf-8")

print("Starting smoke test...")
result = subprocess.run(["uv", "run", "python", "train.py", "--config", str(smoke_config_path)], capture_output=True, text=True)
if result.returncode != 0:
    print(result.stdout[-1500:], result.stderr[-1500:])
    raise RuntimeError("Smoke test failed. Check the messages above.")

smoke = json.loads((STORAGE_ROOT / "results" / "smoke" / "IVAN_SMOKE" / "summary.json").read_text(encoding="utf-8"))
print("Device used:", smoke["device"])
print("Steps:", smoke["total_steps"], "| optimizer updates:", smoke["optimizer_updates"])
print("Model saved:", smoke["model_path"])
assert smoke["optimizer_updates"] > 0, "No learning updates happened."
print("SUCCESS: the training pipeline works (this is a pipeline check, not evidence of learning).")

# 8. The Ten Hyperparameter Experiments

Each experiment trains a brand-new DQN for **100,000 environment steps** with the same seed (42), the same game and the same network. Only the hyperparameters differ.

**What happened in practice:** I already trained all ten on my own computer (CPU, between 33 and 68 minutes per run). Their models, logs and plots are saved in the repository under `models/ivan`, `results/ivan` and `plots/ivan`.

By default this notebook **loads those finished results**, so it runs in minutes. If you want to train again, set `RETRAIN = True`. Finished experiments are skipped, so to retrain from scratch first move the old `results/ivan` and `models/ivan` folders aside. A full retrain takes several hours.

In [ ]:
# Load the finished experiments, or (optionally) train the missing ones
RETRAIN = False                      # True = train experiments that have no results yet
TOTAL_STEPS = 100_000
SEED = 42
MAX_EXPERIMENTS_THIS_SESSION = None  # e.g. 3 to stop after three runs

SRC_RESULTS = REPO_DIR / "results" / "ivan"
SRC_MODELS = REPO_DIR / "models" / "ivan"
SRC_PLOTS = REPO_DIR / "plots" / "ivan"

def is_finished(experiment_id):
    summary_file = SRC_RESULTS / experiment_id / "summary.json"
    model_file = SRC_MODELS / f"{experiment_id}.pth"
    if not (summary_file.exists() and model_file.exists()):
        return False
    return json.loads(summary_file.read_text(encoding="utf-8")).get("total_steps") == TOTAL_STEPS

if RETRAIN:
    os.environ["MPLBACKEND"] = "Agg"
    trained_now = 0
    for index, experiment in enumerate(EXPERIMENTS, start=1):
        experiment_id = experiment["experiment_id"]
        if is_finished(experiment_id):
            print(f"[{index}/10] {experiment_id} already finished. Skipping.")
            continue
        config = {**experiment, "max_steps": TOTAL_STEPS, "seed": SEED, "device": "auto", "output_root": str(REPO_DIR)}
        config_path = Path(tempfile.gettempdir()) / f"{experiment_id}_config.json"
        config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
        print(f"\n=== STARTING {experiment_id} ({index}/10) ===")
        started = time.time()
        subprocess.run(["uv", "run", "python", "train.py", "--config", str(config_path)], check=True)
        print(f"{experiment_id} done in {(time.time() - started) / 60:.1f} minutes")
        trained_now += 1
        if MAX_EXPERIMENTS_THIS_SESSION is not None and trained_now >= MAX_EXPERIMENTS_THIS_SESSION:
            print("Session limit reached. Remaining experiments were not started.")
            break
else:
    print("RETRAIN is False: loading my finished results from the repository.")

missing = [e["experiment_id"] for e in EXPERIMENTS if not is_finished(e["experiment_id"])]
print("Finished experiments:", 10 - len(missing), "/ 10")
assert not missing, f"These experiments have no finished results yet: {missing}"

# 9. Training Results

For each experiment this table shows what happened **during training**.

The most useful number is the **mean reward of the last 10 training episodes**. Be careful with it: during training the agent still makes some random moves (epsilon goes from 1.0 down to 0.1 over the whole run), and 10 games is a small sample, so this number is noisy. That is why section 10 tests every model again with no randomness.

In [ ]:
# Build the training results table and save it as CSV
OUT_RESULTS = REPO_DIR / "results" / "ivan"
OUT_PLOTS = REPO_DIR / "plots" / "ivan"
OUT_RESULTS.mkdir(parents=True, exist_ok=True)
OUT_PLOTS.mkdir(parents=True, exist_ok=True)

training_rows = []
for experiment in EXPERIMENTS:
    summary = json.loads((SRC_RESULTS / experiment["experiment_id"] / "summary.json").read_text(encoding="utf-8"))
    training_rows.append({
        "Experiment": experiment["experiment_id"],
        "Learning Rate": experiment["learning_rate"],
        "Gamma": experiment["gamma"],
        "Batch Size": experiment["batch_size"],
        "Epsilon Start": experiment["epsilon_start"],
        "Epsilon End": experiment["epsilon_end"],
        "Epsilon Decay": experiment["epsilon_decay"],
        "Training Steps": summary["total_steps"],
        "Episodes": summary["completed_episodes"],
        "Optimizer Updates": summary["optimizer_updates"],
        "Last 10 Mean Reward": round(summary["last_10_episode_mean_reward"], 2),
        "Last 10 Mean Length": round(summary["last_10_episode_mean_length"], 1),
        "Best Episode Reward": summary["best_episode_reward"],
        "Training Time (min)": round(summary["elapsed_seconds"] / 60, 1),
        "Device": summary["device"],
    })

def save_csv(rows, path):
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

training_csv_path = OUT_RESULTS / "ivan_training_results.csv"
save_csv(training_rows, training_csv_path)
show_table(training_rows, ["Experiment", "Learning Rate", "Gamma", "Batch Size", "Epsilon Start", "Epsilon End",
                           "Epsilon Decay", "Episodes", "Last 10 Mean Reward", "Best Episode Reward", "Training Time (min)", "Device"])
print("Saved:", training_csv_path)

# 10. Evaluate Every Saved Model with the Same Games

A fair comparison needs the same test for every model. So each of the ten models plays the **same 10 games** (game seeds 1000 to 1009) and always picks its best-looking move (epsilon = 0, no randomness).

This is done by my script `evaluate_experiments.py`. If its result file already exists, it is reused. The standard deviation shows how much the score changes from game to game.

In [ ]:
# Evaluate all ten models (or reuse the saved evaluation)
eval_json_path = SRC_RESULTS / "evaluation.json"

if not eval_json_path.exists():
    print("No saved evaluation found. Running evaluate_experiments.py (a few minutes)...")
    subprocess.run(["uv", "run", "python", "evaluate_experiments.py", "--episodes", "10", "--seed", "1000"], check=True)
else:
    print("Reusing the saved evaluation:", eval_json_path)

evaluation = json.loads(eval_json_path.read_text(encoding="utf-8"))
EVAL_EPISODES, EVAL_SEED = evaluation["episodes"], evaluation["base_seed"]
print(f"Each model played {EVAL_EPISODES} games with seeds {EVAL_SEED} to {EVAL_SEED + EVAL_EPISODES - 1}.")

episode_rows, evaluation_rows = [], []
for experiment_id in ids:
    result = evaluation["results"][experiment_id]
    for i, reward in enumerate(result["rewards"]):
        episode_rows.append({"Experiment": experiment_id, "Episode": i + 1, "Seed": EVAL_SEED + i, "Reward": reward})
    evaluation_rows.append({
        "Experiment": experiment_id,
        "Evaluation Mean Reward": round(result["mean"], 2),
        "Evaluation Std Reward": round(result["std"], 2),
        "Minimum Reward": min(result["rewards"]),
        "Maximum Reward": max(result["rewards"]),
    })

episodes_csv_path = OUT_RESULTS / "ivan_evaluation_episode_results.csv"
summary_csv_path = OUT_RESULTS / "ivan_evaluation_summary.csv"
save_csv(episode_rows, episodes_csv_path)
save_csv(evaluation_rows, summary_csv_path)
show_table(evaluation_rows, list(evaluation_rows[0].keys()))
print("Saved:", episodes_csv_path.name, "and", summary_csv_path.name)

# 11. Combine Training and Evaluation, Then Rank

The final ranking uses the **evaluation** reward, because it is measured the same way for every model. The training reward is kept next to it for comparison.

In [ ]:
# Merge training and evaluation, rank by evaluation reward
training_by_id = {row["Experiment"]: row for row in training_rows}
final_rows = []
for row in evaluation_rows:
    merged = {**training_by_id[row["Experiment"]], **row}
    final_rows.append(merged)

final_rows.sort(key=lambda r: r["Evaluation Mean Reward"], reverse=True)
for rank, row in enumerate(final_rows, start=1):
    row["Evaluation Rank"] = rank

final_csv_path = OUT_RESULTS / "ivan_final_combined_results.csv"
save_csv(final_rows, final_csv_path)
show_table(final_rows, ["Evaluation Rank", "Experiment", "Learning Rate", "Gamma", "Batch Size",
                        "Last 10 Mean Reward", "Evaluation Mean Reward", "Evaluation Std Reward"])

best = final_rows[0]
best_experiment = best["Experiment"]
print(f"Best by evaluation: {best_experiment} ({best['Evaluation Mean Reward']}), "
      f"settings: lr={best['Learning Rate']}, gamma={best['Gamma']}, batch={best['Batch Size']}")
print("Saved:", final_csv_path)

# 12. Plots Across All Ten Experiments

The training script already saved one plot per experiment (reward, episode length and loss) in `plots/ivan`. Here are four plots that compare **all ten experiments** at once:

1. training reward (last 10 episodes),
2. evaluation reward (with the spread between games),
3. training reward next to evaluation reward,
4. how the training reward grew over time.

In [ ]:
# Plot 1: training reward per experiment
import matplotlib.pyplot as plt

by_training = sorted(training_rows, key=lambda r: r["Last 10 Mean Reward"], reverse=True)
plt.figure(figsize=(12, 6))
plt.bar([r["Experiment"] for r in by_training], [r["Last 10 Mean Reward"] for r in by_training])
plt.title("Training Performance Across Ten DQN Experiments")
plt.xlabel("Experiment")
plt.ylabel("Last 10 Training Episodes: Mean Reward")
plt.xticks(rotation=45)
plt.tight_layout()
training_plot_path = OUT_PLOTS / "ivan_training_rewards_comparison.png"
plt.savefig(training_plot_path, dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Plot 2: evaluation reward per experiment (bars show the average, lines show the spread between games)
plt.figure(figsize=(12, 6))
plt.bar([r["Experiment"] for r in final_rows], [r["Evaluation Mean Reward"] for r in final_rows],
        yerr=[r["Evaluation Std Reward"] for r in final_rows], capsize=4, color="orange")
plt.title("Evaluation Performance Across Ten DQN Experiments (10 games each, no random moves)")
plt.xlabel("Experiment")
plt.ylabel("Mean Reward (lines: one standard deviation)")
plt.xticks(rotation=45)
plt.tight_layout()
evaluation_plot_path = OUT_PLOTS / "ivan_evaluation_rewards_comparison.png"
plt.savefig(evaluation_plot_path, dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Plot 3: training reward next to evaluation reward
ordered = sorted(final_rows, key=lambda r: r["Experiment"])
x = range(len(ordered))
width = 0.38
plt.figure(figsize=(14, 7))
plt.bar([i - width / 2 for i in x], [r["Last 10 Mean Reward"] for r in ordered], width=width, label="Training: last 10 episodes")
plt.bar([i + width / 2 for i in x], [r["Evaluation Mean Reward"] for r in ordered], width=width, label="Evaluation: 10 games")
plt.title("Training and Evaluation Reward Comparison")
plt.xlabel("Experiment")
plt.ylabel("Mean Reward")
plt.xticks(list(x), [r["Experiment"] for r in ordered], rotation=45)
plt.legend()
plt.tight_layout()
comparison_plot_path = OUT_PLOTS / "ivan_training_vs_evaluation.png"
plt.savefig(comparison_plot_path, dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Plot 4: how the training reward grew over time (moving average of 20 episodes)
def moving_average(values, window=20):
    return [sum(values[max(0, i - window + 1): i + 1]) / len(values[max(0, i - window + 1): i + 1]) for i in range(len(values))]

plt.figure(figsize=(12, 6))
for experiment_id in ids:
    with (SRC_RESULTS / experiment_id / "episodes.csv").open(encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    steps = [int(r["total_steps"]) for r in rows]
    rewards = [float(r["reward"]) for r in rows]
    style = {"linewidth": 3} if experiment_id == best_experiment else {"linewidth": 1, "alpha": 0.7}
    plt.plot(steps, moving_average(rewards), label=experiment_id, **style)
plt.title("Training Reward Over Time (moving average of 20 episodes; thick line = best in evaluation)")
plt.xlabel("Training step")
plt.ylabel("Reward")
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
curves_plot_path = OUT_PLOTS / "ivan_training_curves.png"
plt.savefig(curves_plot_path, dpi=200, bbox_inches="tight")
plt.show()

# 13. Gameplay Demonstration

This is the **best agent** (the top model in the ranking above) playing one full game with no random moves.

The clip was made by `record_demo.py`: the model played 10 games with the same seeds as the evaluation, and the game with the highest score was recorded. It is the best of 10, so it looks better than the average game.

In [ ]:
# Show the saved gameplay clip (set RECORD_DEMO = True to record a new one for the best model)
RECORD_DEMO = False
demo_gif = REPO_DIR / "demo" / f"ivan_best_agent_{best_experiment.replace('TASK3_', '')}.gif"
demo_mp4 = demo_gif.with_suffix(".mp4")

if RECORD_DEMO or not demo_gif.exists():
    print("Recording the demo for", best_experiment, "...")
    subprocess.run(["uv", "run", "python", "record_demo.py", "--model", str(SRC_MODELS / f"{best_experiment}.pth"),
                    "--output", str(demo_mp4), "--gif", str(demo_gif)], check=True)

print("Video file:", demo_mp4)
try:
    from IPython.display import Image, display
    display(Image(filename=str(demo_gif)))
except ImportError:
    print("GIF saved at:", demo_gif)

# 14. Written Results and Interpretation

The report below is my results table from the repository. It holds the numbers for every experiment, a short written observation for each one, and my interpretation. The observations come from the real logs, not from guesses.

In [ ]:
# Show my results report
report_path = REPO_DIR / "IvanHyperparameter_experiment_results.md"
report_text = report_path.read_text(encoding="utf-8")
try:
    from IPython.display import Markdown, display
    display(Markdown(report_text))
except ImportError:
    print(report_text)

# 15. What I Learned and Limits of These Results

- **All ten agents improved** during training, but training reward is a noisy way to compare them.
- **Best in evaluation:** the setting with the highest evaluation score is shown in section 11 (it was `TASK3_EXP04`, gamma 0.95, in my run). Its lead over the next models is clear but not huge, because scores change a lot from game to game.
- **Training vs evaluation:** some runs looked good in training and weak in evaluation (for example `TASK3_EXP02`). This is why the evaluation step matters.
- **Limits:** each setting was trained **once** (one seed) for only **100,000 steps** on a CPU, and tested on 10 games. The differences between the middle experiments are within the noise. A fairer test would repeat each setting with several seeds and train for longer.

# 16. Build the Final Submission Folder

Everything needed for the hand-in is copied into one folder and one ZIP file: the result tables, the plots, the ten models, the written report and the gameplay clip. In Colab this goes to Google Drive, so it stays after the session ends.

In [ ]:
# Collect the final files into a folder and a ZIP
SUBMISSION_FOLDER = STORAGE_ROOT / "Ivan_DQN_Final_Submission"
if SUBMISSION_FOLDER.exists():
    shutil.rmtree(SUBMISSION_FOLDER)

parts = {
    "results": [training_csv_path, episodes_csv_path, summary_csv_path, final_csv_path, report_path,
                REPO_DIR / "ivan_experiments.json", REPO_DIR / "ivan_observations.json", eval_json_path],
    "plots": [training_plot_path, evaluation_plot_path, comparison_plot_path, curves_plot_path] + sorted(SRC_PLOTS.glob("TASK3_EXP*.png")),
    "models": sorted(SRC_MODELS.glob("TASK3_EXP*.pth")),
    "demo": [p for p in [demo_gif, demo_mp4] if p.exists()],
}
for folder, files in parts.items():
    (SUBMISSION_FOLDER / folder).mkdir(parents=True, exist_ok=True)
    for file_path in files:
        if not Path(file_path).exists():
            raise FileNotFoundError(file_path)
        shutil.copy2(file_path, SUBMISSION_FOLDER / folder / Path(file_path).name)

zip_path = STORAGE_ROOT / "Ivan_DQN_Final_Submission.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in SUBMISSION_FOLDER.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(SUBMISSION_FOLDER.parent))

print("FINAL SUBMISSION FOLDER:", SUBMISSION_FOLDER)
for folder in parts:
    print(f"- {folder}/: {len(list((SUBMISSION_FOLDER / folder).iterdir()))} files")
print("ZIP archive:", zip_path, f"({zip_path.stat().st_size / (1024 * 1024):.1f} MB)")